<a href="https://colab.research.google.com/github/vt214-cyber/project2-tcga-p53-capstone-project/blob/main/01_cohort_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Notebook 01 — Clean the cohort, describe the patients

**Produces:** a clean patient table, and your "Table 1".

---

> **This notebook stands alone.** Nothing in Notebook 02 uses its
> output. If you get stuck here, go to `02_p53_analysis` — it has its
> own, already-clean data file.

## Why you are doing this

Open any clinical paper and the first table is always the same: *who
were the patients?* How many, how long were they followed, how many
died. It is called **Table 1**, and no reviewer reads past it if it is
wrong.

You have the raw patient export exactly as it comes out of the database
— barcodes instead of patient IDs, numeric codes instead of sample
labels, follow-up time in days, and missing values written four
different ways. **Turn that into something you could put Table 1 on.**

> ### ⚠️ Run every cell, in order, from the top
>
> Your functions only exist once you have **run** the cell that defines
> them. If you edit a function, you must re-run its cell or Python
> carries on using the old version — which is the single most confusing
> bug in notebook work.
>
> If anything stops making sense, use **Runtime → Restart and run all**
> (Colab) or **Kernel → Restart & Run All** (VS Code) and start clean.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 66 of 89** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [ ]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


STUDENT_ID = "vt214"   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

MY_PHENOTYPE = fetch(f"phenotype_raw_{CANCER_TYPE}.csv")
EXAMPLE_PHENOTYPE = fetch("phenotype_raw_example.csv")

# Given to you - the codes used in the raw export.
SAMPLE_TYPES = {
    "01": "Primary Tumor",
    "02": "Recurrent Tumor",
    "03": "Primary Blood Derived Cancer - Peripheral Blood",
    "05": "Additional - New Primary",
    "06": "Metastatic",
    "07": "Additional Metastatic",
    "11": "Solid Tissue Normal",
}

# Given to you - the four ways this export writes "missing".
MISSING = ["not reported", "--", ""]

raw = pd.read_csv(MY_PHENOTYPE)
print(raw.shape)
raw.head()

vt214: your gene is AEN, your cancer type is LUSC
(563, 5)


,sampleID,primary disease or tissue,sample_type_code,days_to_last_followup,vital_status
0,TCGA-98-A53C-01,LUNG SQUAMOUS CELL CARCINOMA,1,822.0,Alive
1,TCGA-39-5029-01,lung squamous cell carcinoma,1,740.0,Dead
2,TCGA-66-2758-01,lung squamous cell carcinoma,1,639.0,Alive
3,TCGA-85-A50M-01,lung squamous cell carcinoma,1,826.0,Dead
4,TCGA-98-A53B-01,Lung Squamous Cell Carcinoma,1,61.0,Dead


## Look at the mess before you touch it

Never start cleaning before you have looked.

In [ ]:
# DEMO - what is actually in each column?
for col in raw.columns:
    print(f"--- {col}")
    print(raw[col].astype(str).str.strip().value_counts().head(4).to_string())
    print()

--- sampleID
sampleID
TCGA-NC-A5HH-01    2
TCGA-77-6845-01    2
TCGA-46-3765-01    2
TCGA-56-8201-11    2

--- primary disease or tissue
primary disease or tissue
LUNG SQUAMOUS CELL CARCINOMA    190
lung squamous cell carcinoma    190
Lung Squamous Cell Carcinoma    183

--- sample_type_code
sample_type_code
1     511
11     52

--- days_to_last_followup
days_to_last_followup
nan             22
not reported    19
--              18
61.0             8

--- vital_status
vital_status
Alive    309
Dead     254



### What you should have spotted

1. **`sampleID` is a TCGA barcode**, like `TCGA-05-4244-01`. The first
   **12 characters** are the patient — `TCGA-05-4244`. One patient can
   appear more than once.
2. **`primary disease or tissue` is inconsistent** — upper case, lower
   case, stray spaces. Three spellings of one thing is three categories
   to pandas.
3. **`sample_type_code` is numeric.** `01` is a primary tumour, `11` is
   normal tissue, `06` a metastasis. `SAMPLE_TYPES` translates them.
4. **`days_to_last_followup` is not a number.** Some rows say
   `not reported`, some `--`, some are blank. Leave them and the column
   stays text, and you cannot take a median of text.
5. **There are duplicate rows.** The same `sampleID` appears twice.

## Task 1a — `load_and_clean`

**Takes** `path` — the path to a raw phenotype CSV.

**Returns** a DataFrame with **exactly these columns, in this order**:

```
sample_id, patient_id, cancer_type, sample_type, is_tumour, followup_years, vital_status
```

| column | what it must contain |
|---|---|
| `sample_id` | the barcode, unchanged |
| `patient_id` | first 12 characters of the barcode |
| `cancer_type` | stripped of spaces, upper case |
| `sample_type` | the readable label, via `SAMPLE_TYPES` |
| `is_tumour` | `True` / `False`, a real boolean — everything except `Solid Tissue Normal` |
| `followup_years` | days ÷ 365.25, rounded to 1 dp, `NaN` where missing |
| `vital_status` | `Alive`, `Dead`, or `NaN` |

Drop rows with a duplicated `sample_id`, keeping the first, and reset
the index.

> 💡 Do not start by filling in the function. Add a cell, load the file,
> and work out the seven columns one at a time in raw code. Wrap it up
> only once it works — see *How to build each function* in
> `00_start_here`.

**Three things that catch people out:** the sentinels must become `NaN`
*before* you convert to a number; `followup_years` must end up a float,
not text; and the column order is part of the contract.

In [ ]:
# Showing the first five rows of data
raw.head()

,sampleID,primary disease or tissue,sample_type_code,days_to_last_followup,vital_status
0,TCGA-98-A53C-01,LUNG SQUAMOUS CELL CARCINOMA,1,822.0,Alive
1,TCGA-39-5029-01,lung squamous cell carcinoma,1,740.0,Dead
2,TCGA-66-2758-01,lung squamous cell carcinoma,1,639.0,Alive
3,TCGA-85-A50M-01,lung squamous cell carcinoma,1,826.0,Dead
4,TCGA-98-A53B-01,Lung Squamous Cell Carcinoma,1,61.0,Dead


In [ ]:
# Shows the names of the columns in file
raw.columns

Index(['sampleID', 'primary disease or tissue', 'sample_type_code',
       'days_to_last_followup', 'vital_status'],
      dtype='object')

In [ ]:
# Check the data type of each column before cleaning
raw.dtypes

,0
sampleID,object
primary disease or tissue,object
sample_type_code,int64
days_to_last_followup,object
vital_status,object


In [ ]:
# Check the original sample barcode
raw["sampleID"].head()

,sampleID
0,TCGA-98-A53C-01
1,TCGA-39-5029-01
2,TCGA-66-2758-01
3,TCGA-85-A50M-01
4,TCGA-98-A53B-01


In [ ]:
# Extract the first 12 characters to identify each patient
raw["sampleID"].str[:12].head()

,sampleID
0,TCGA-98-A53C
1,TCGA-39-5029
2,TCGA-66-2758
3,TCGA-85-A50M
4,TCGA-98-A53B


In [ ]:
# Remove extra spaces and convert cancer types to uppercase
raw["primary disease or tissue"].str.strip().str.upper().head()

,primary disease or tissue
0,LUNG SQUAMOUS CELL CARCINOMA
1,LUNG SQUAMOUS CELL CARCINOMA
2,LUNG SQUAMOUS CELL CARCINOMA
3,LUNG SQUAMOUS CELL CARCINOMA
4,LUNG SQUAMOUS CELL CARCINOMA


In [ ]:
# Convert numeric codes into readable labels using SAMPLE_TYPES
sample_codes = pd.to_numeric(raw["sample_type_code"], errors="coerce")
sample_codes = sample_codes.astype("Int64").astype("string")
sample_codes = sample_codes.str.zfill(2)
sample_codes.replace(SAMPLE_TYPES).head()

,sample_type_code
0,Primary Tumor
1,Primary Tumor
2,Primary Tumor
3,Primary Tumor
4,Primary Tumor


In [ ]:
SAMPLE_TYPES

{'01': 'Primary Tumor',
 '02': 'Recurrent Tumor',
 '03': 'Primary Blood Derived Cancer - Peripheral Blood',
 '05': 'Additional - New Primary',
 '06': 'Metastatic',
 '07': 'Additional Metastatic',
 '11': 'Solid Tissue Normal'}

In [ ]:
# Count how many samples belong to each sample type
sample_codes.replace(SAMPLE_TYPES).value_counts()

,count
sample_type_code,
Primary Tumor,511
Solid Tissue Normal,52


In [ ]:
# Identify tumour samples and distinguish them from normal tissues
sample_type = sample_codes.replace(SAMPLE_TYPES)
is_tumour = (sample_type != "Solid Tissue Normal").astype(bool)
is_tumour.head()

,sample_type_code
0,True
1,True
2,True
3,True
4,True


In [ ]:
# Convert followup days into years and handle missing value
followup = raw["days_to_last_followup"].astype("string")
followup = followup.str.strip().str.lower()
# Replace missing value markers with NaN
followup = followup.replace(["not reported", "--", ""], np.nan)
# Convert the cleaned values into numbers
followup = pd.to_numeric(followup, errors="coerce")
# Divide by 365.25 and round to one decimal place
followup_years = (followup / 365.25).round(1)
followup_years.head()

,days_to_last_followup
0,2.3
1,2.0
2,1.7
3,2.3
4,0.2


In [ ]:
# Standardise vital status values as Alive or Dead
status = raw["vital_status"].astype("string")
status = status.str.strip().str.lower()
vital_status = status.replace({
     "alive": "Alive", "dead": "Dead"
     })
vital_status.head()

,vital_status
0,Alive
1,Dead
2,Alive
3,Dead
4,Dead


In [ ]:
# Remove duplicate sample IDs while keeping the first
raw_no_duplicates = raw.drop_duplicates( subset="sampleID", keep="first")
raw_no_duplicates.shape

(548, 5)

In [ ]:
# Check that no duplicate sample IDs remain
raw_no_duplicates["sampleID"].duplicated().sum()

np.int64(0)

In [ ]:
def load_and_clean(path):
    """
    Load and clean the raw TCGA phenotype data

    Parameters:
        path: The file path of the raw phenotype CSV file.

    Returns:
        A cleaned DataFrame containing seven columns:
        sample_id, patient_id, cancer_type, sample_type, is_tumour, followup_years and vital_status.
    """
    # Step 1: Read data from CSV file
    df = pd.read_csv(path)

    # Step 2: Remove the duplicate sample IDs keeping the first
    df = df.drop_duplicates(subset='sampleID' , keep='first')

    # Step 3: Create an empty dataframe to store clean columns
    clean = pd.DataFrame()

    # Step 4: Keep the original sample barcode
    clean['sample_id'] = df['sampleID']
    # Extract the first 12 characters to get the patient ID
    clean['patient_id'] = clean['sample_id'].str[:12]

    # Step 5: Clean the cancer type and convert cancer types to uppercase
    clean["cancer_type"] = df["primary disease or tissue"]
    clean["cancer_type"] = clean["cancer_type"].str.strip() #removing
    clean["cancer_type"] = clean["cancer_type"].str.upper() #uppercase

    # Step 6: Convert sample type codes into readable labels
    sample_codes = pd.to_numeric(df["sample_type_code"], errors="coerce")
    # Convert codes into two digit strings
    sample_codes = sample_codes.astype("Int64").astype("string")
    sample_codes = sample_codes.str.zfill(2)
    # Translate codes into readable sample types
    clean["sample_type"] = sample_codes.replace(SAMPLE_TYPES)
    #Identify tumour samples - everything except Solid Tissue Normal is classed as tumour
    clean["is_tumour"] = (clean["sample_type"] != "Solid Tissue Normal").astype(bool)

    # Step 7: Clean follow up values
    followup = df["days_to_last_followup"]

    followup = followup.astype("string")
    followup = followup.str.strip() #removing
    followup = followup.str.lower() #lowercase

    # Replace missing value descriptions with NaN before converting to numbers
    followup = followup.replace(["not reported", "--", ""], np.nan)
    # Convert follow up values into numbers
    followup = pd.to_numeric(followup, errors="coerce")
    # Convert days into years and round to one decimal place
    clean["followup_years"] = (followup / 365.25).round(1).astype(float)

    # Step 8: Clean vital status and standardise the labels
    status = df["vital_status"]
    status = status.astype("string")
    status = status.str.strip() #removing
    status = status.str.lower() #lowercase

    # Replace missing value descriptions with NaN
    status = status.replace(["not reported", "--", ""], np.nan)

    clean["vital_status"] = status.replace({
        "alive": "Alive",
        "dead": "Dead"
        })

    # Step 9: Select columns in the required order
    clean = clean[
        [
            "sample_id",
            "patient_id",
            "cancer_type",
            "sample_type",
            "is_tumour",
            "followup_years",
            "vital_status"
            ]
        ]

    # Step 10: Reset the Index so rows are numbered from zero
    clean = clean.reset_index(drop=True)

    # Return the finished cleaned DataFrame
    return clean

In [ ]:
# CHECK - runs your function on the worked example (CDKN1A in BRCA)
EXPECTED_COLUMNS = ["sample_id", "patient_id", "cancer_type",
                    "sample_type", "is_tumour", "followup_years",
                    "vital_status"]

checked = load_and_clean(EXAMPLE_PHENOTYPE)

run_checks({
    "returns a DataFrame":
        lambda: isinstance(checked, pd.DataFrame),
    "columns are correct, and in the right order":
        lambda: list(checked.columns) == EXPECTED_COLUMNS,
    "patient_id is the first 12 characters of the barcode":
        lambda: (checked["sample_id"].str[:12] == checked["patient_id"]).all(),
    "duplicate sample_id rows were dropped":
        lambda: checked["sample_id"].is_unique,
    "no sentinel strings survived":
        lambda: not any((checked.astype(str) == s).any().any()
                        for s in MISSING),
    "followup_years is a float, not text":
        lambda: checked["followup_years"].dtype == float,
    "is_tumour is a real boolean":
        lambda: checked["is_tumour"].dtype == bool,
    "row count matches the worked example (1212)":
        lambda: len(checked) == 1212,
    "tumour/normal split matches (1099/113)":
        lambda: (int(checked["is_tumour"].sum()) == 1099
                 and int((~checked["is_tumour"]).sum()) == 113),
})

✅ returns a DataFrame
✅ columns are correct, and in the right order
✅ patient_id is the first 12 characters of the barcode
✅ duplicate sample_id rows were dropped
✅ no sentinel strings survived
✅ followup_years is a float, not text
✅ is_tumour is a real boolean
✅ row count matches the worked example (1212)
✅ tumour/normal split matches (1099/113)

9 of 9 checks passed


Once that passes, run it on your own data:

In [ ]:
clean = load_and_clean(MY_PHENOTYPE)
clean.head()

,sample_id,patient_id,cancer_type,sample_type,is_tumour,followup_years,vital_status
0,TCGA-98-A53C-01,TCGA-98-A53C,LUNG SQUAMOUS CELL CARCINOMA,Primary Tumor,True,2.3,Alive
1,TCGA-39-5029-01,TCGA-39-5029,LUNG SQUAMOUS CELL CARCINOMA,Primary Tumor,True,2.0,Dead
2,TCGA-66-2758-01,TCGA-66-2758,LUNG SQUAMOUS CELL CARCINOMA,Primary Tumor,True,1.7,Alive
3,TCGA-85-A50M-01,TCGA-85-A50M,LUNG SQUAMOUS CELL CARCINOMA,Primary Tumor,True,2.3,Dead
4,TCGA-98-A53B-01,TCGA-98-A53B,LUNG SQUAMOUS CELL CARCINOMA,Primary Tumor,True,0.2,Dead


## Task 1b — `describe_cohort`

Now the actual deliverable.

**Takes** `clean` — a table in the shape `load_and_clean` returns.

**Returns** a dict of the six numbers that make up Table 1:

```python
{"n_patients": int, "n_samples": int, "n_tumour": int,
 "n_normal": int, "median_followup_years": float, "n_deceased": int}
```

`n_patients` and `n_samples` are **not** the same number, and noticing
why is half the point of this task.

In [ ]:
# Counting the number of different patients in the cohort
clean["patient_id"].nunique()

498

In [ ]:
# Count the total number of samples in the cohort
len(clean)

548

In [ ]:
# Count the number of tumour samples
clean["is_tumour"].sum()

np.int64(498)

In [ ]:
# Count tumour and normal samples
clean["is_tumour"].value_counts()

,count
is_tumour,
True,498
False,50


In [ ]:
# Check for missing values in the tumour column
clean["is_tumour"].isna().sum()

np.int64(0)

In [ ]:
# Calculate the median followup time in years
clean["followup_years"].median()

1.8

In [ ]:
# Count records where vital status is Dead
clean["vital_status"].value_counts()

,count
vital_status,
Alive,302
Dead,246


In [ ]:
# Count the number of different deceased patients
clean.loc[clean["vital_status"] == "Dead"]
len(deceased)

246

In [ ]:
# Count all tumour and normal values
clean["is_tumour"].value_counts(dropna=False)

,count
is_tumour,
True,498
False,50


In [ ]:
def describe_cohort(clean):
    """
    Calculate summary statistics for the cleaned cohort.
    Returns a dictionary containing the following statistics:
    """
    # Step 1: Count the number of different patients
    n_patients = clean["patient_id"].nunique()

    # Step 2: Count the total number of samples
    n_samples = len(clean)

    # Step 3: Count tumour samples
    n_tumour = clean["is_tumour"].sum()

    # Step 4: Count normal samples
    n_normal = len(clean) - n_tumour

    # Step 5:Calculate the median followup time in years
    median_followup_years = clean["followup_years"].median()

    # Step 6: Count records where vital status is dead
    n_deceased = clean.loc[clean["vital_status"] == "Dead"]
    n_deceased = len(n_deceased)

    # Return all six statistics in a dictionary
    return {
        "n_patients": int(n_patients),
        "n_samples": int(n_samples),
        "n_tumour": int(n_tumour),
        "n_normal": int(n_normal),
        "median_followup_years": float(median_followup_years),
        "n_deceased": int(n_deceased)
    }



In [ ]:
summary = describe_cohort(clean)
summary

{'n_patients': 498,
 'n_samples': 548,
 'n_tumour': 498,
 'n_normal': 50,
 'median_followup_years': 1.8,
 'n_deceased': 246}

In [ ]:
checked.shape

(1212, 7)

In [ ]:
describe_cohort(checked)

{'n_patients': 1093,
 'n_samples': 1212,
 'n_tumour': 1099,
 'n_normal': 113,
 'median_followup_years': 2.5,
 'n_deceased': 198}

In [ ]:
# CHECK - against the worked example
EXPECTED_COHORT = {
    "n_patients": 1093,
    "n_samples": 1212,
    "n_tumour": 1099,
    "n_normal": 113,
    "median_followup_years": 2.5,
    "n_deceased": 198,
}

summary = describe_cohort(checked)

run_checks({
    "returns a dict":
        lambda: isinstance(summary, dict),
    "has exactly the six required keys":
        lambda: set(summary) == set(EXPECTED_COHORT),
    "samples split into tumour and normal with none left over":
        lambda: summary["n_tumour"] + summary["n_normal"] == summary["n_samples"],
    "there are no more patients than samples":
        lambda: summary["n_patients"] <= summary["n_samples"],
    "every value matches the worked example":
        lambda: all(summary[k] == v for k, v in EXPECTED_COHORT.items()),
})

✅ returns a dict
✅ has exactly the six required keys
✅ samples split into tumour and normal with none left over
✅ there are no more patients than samples
✅ every value matches the worked example

5 of 5 checks passed


In [ ]:
my_cohort = describe_cohort(clean)
my_cohort

{'n_patients': 498,
 'n_samples': 548,
 'n_tumour': 498,
 'n_normal': 50,
 'median_followup_years': 1.8,
 'n_deceased': 246}

In [ ]:
# Confirming usage of correct file
print(MY_PHENOTYPE)
print(EXAMPLE_PHENOTYPE)

phenotype_raw_LUSC.csv
phenotype_raw_example.csv


## Write your Table 1

Use your own numbers. This is marked on whether it is **correct and
readable**, not on how you computed it.

### Table 1 — Summary of the LUSC Cohort

|Characteristics | Values |
|---|---|
| Patients |498 |
| Samples |548 |
| Primary tumour / normal |498 / 50 |
| Median follow-up (years) | 1.8|
| Deceased | 246|

*One or two sentences describing the cohort — and say why the patient
count and the sample count differ.*

The cohort consisted of 498 patients and 548 samples from patients with lung squamous cell carcinoma. There were 498 primary tumour samples and 50 normal tissue samples, with a median followup time of 1.8 years and 246 records marked as deceased.

The sample count is higher than the patient count because some patients contributed more than one sample to the dataset.